In [1]:
import json
import ollama
import re
import random

In [2]:
"""Read json files"""
with open("mbpp/data.json") as data_file:
    data = json.load(data_file)
data[0]

{'text': 'Write a function to find the minimum cost path to reach (m, n) from (0, 0) for the given cost matrix cost[][] and a position (m, n) in cost[][].',
 'code': 'R = 3\r\nC = 3\r\ndef min_cost(cost, m, n): \r\n\ttc = [[0 for x in range(C)] for x in range(R)] \r\n\ttc[0][0] = cost[0][0] \r\n\tfor i in range(1, m+1): \r\n\t\ttc[i][0] = tc[i-1][0] + cost[i][0] \r\n\tfor j in range(1, n+1): \r\n\t\ttc[0][j] = tc[0][j-1] + cost[0][j] \r\n\tfor i in range(1, m+1): \r\n\t\tfor j in range(1, n+1): \r\n\t\t\ttc[i][j] = min(tc[i-1][j-1], tc[i-1][j], tc[i][j-1]) + cost[i][j] \r\n\treturn tc[m][n]',
 'task_id': 1,
 'test_setup_code': '',
 'test_list': ['assert min_cost([[1, 2, 3], [4, 8, 2], [1, 5, 3]], 2, 2) == 8',
  'assert min_cost([[2, 3, 4], [5, 9, 3], [2, 6, 4]], 2, 2) == 12',
  'assert min_cost([[3, 4, 5], [6, 10, 4], [3, 7, 5]], 2, 2) == 16'],
 'challenge_test_list': []}

In [3]:
"""Clean read data"""
def clean_data(data):
    return [{
        'text': elem['text'],
        'code':elem['code'].replace('\r\n', '\n'),
        'task_id':elem['task_id'], 'is_human':1
    } for elem in data]
        
data = clean_data(data)
data[:2]

[{'text': 'Write a function to find the minimum cost path to reach (m, n) from (0, 0) for the given cost matrix cost[][] and a position (m, n) in cost[][].',
  'code': 'R = 3\nC = 3\ndef min_cost(cost, m, n): \n\ttc = [[0 for x in range(C)] for x in range(R)] \n\ttc[0][0] = cost[0][0] \n\tfor i in range(1, m+1): \n\t\ttc[i][0] = tc[i-1][0] + cost[i][0] \n\tfor j in range(1, n+1): \n\t\ttc[0][j] = tc[0][j-1] + cost[0][j] \n\tfor i in range(1, m+1): \n\t\tfor j in range(1, n+1): \n\t\t\ttc[i][j] = min(tc[i-1][j-1], tc[i-1][j], tc[i][j-1]) + cost[i][j] \n\treturn tc[m][n]',
  'task_id': 1,
  'is_human': 1},
 {'text': 'Write a function to find the similar elements from the given two tuple lists.',
  'code': 'def similar_elements(test_tup1, test_tup2):\n  res = tuple(set(test_tup1) & set(test_tup2))\n  return (res) ',
  'task_id': 2,
  'is_human': 1}]

In [4]:
"""Generate LLM data by giving the problem as a prompt"""

def generate_synth_code(prompt):
    response = ollama.chat(
        model="codellama:7b-instruct",
        messages=[
            {'role': 'system', 'content': "You are a code generation assistant. Generate only the Python function code, no explanations. Also, don't add comments to the code."},
            {'role': 'user', 'content': f"Write a Python function:\n{prompt}"}
        ],
        # TODO: Check if these options make sense
        # options={
        #     'temperature': 0.8,
        #     'top_p': 0.95
        # }
    )
    
    pattern = r'```\n(.*?)```'
    code = re.search(pattern, response['message']['content'], re.DOTALL).group(1)
    return code

def add_synth_data(data):
    result = []
    i = 0
    for elem in data:
        result.append(elem)
        result.append({
            'text': elem['text'],
            'code': generate_synth_code(elem['text']),
            'task_id': elem['task_id'],
            'is_human': 0
        })
        i += 1
        if i % 10 == 0:
            print(f"Generated {i} of {len(data)}")

    return result

data = add_synth_data(data)

Generated 10 of 100
Generated 20 of 100
Generated 30 of 100
Generated 40 of 100
Generated 50 of 100
Generated 60 of 100
Generated 70 of 100
Generated 80 of 100
Generated 90 of 100
Generated 100 of 100


In [5]:
"""Strip data fields, randomize and save"""
print(f"Size: {len(data)} samples")
print(f"Human: {sum(1 for d in data if d['is_human'] == 1)}")
print(f"Synthetic: {sum(1 for d in data if d['is_human'] == 0)}")

data = [{'code': elem['code'], 'is_human': elem['is_human']} for elem in data]
random.shuffle(data)

with open("detection_data.json", "w") as data_file:
    json.dump(data, data_file, indent=2)

Size: 200 samples
Human: 100
Synthetic: 100
